In [3]:
from pymongo import MongoClient
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import avg

**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

In [4]:
client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

koleksi_menu.delete_many({})

data_menu = [
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 50},
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 60},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 100},
    {"nama_menu": "Cafe Latte", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 40},
    {"nama_menu": "Cappuccino", "harga": 26000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 45},
    {"nama_menu": "Long Black", "harga": 21000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 30},
    
    {"nama_menu": "Mie Goreng Nusantara", "harga": 15000, "kategori": "Makanan", "pedas_level": 2, "rekomendasi": True},
    {"nama_menu": "Nasi Goreng Gila", "harga": 28000, "kategori": "Makanan", "pedas_level": 3, "stok": 25},
    {"nama_menu": "Cireng Bumbu Rujak", "harga": 12000, "kategori": "Makanan", "pedas_level": 1, "stok": 80},
    {"nama_menu": "Kentang Goreng Keju", "harga": 17000, "kategori": "Makanan", "pedas_level": 0, "stok": 55},
    {"nama_menu": "Roti Bakar Cokelat", "harga": 16000, "kategori": "Makanan", "pedas_level": 0, "stok": 35},
    {"nama_menu": "Ayam Geprek Sambal Korek", "harga": 32000, "kategori": "Makanan", "pedas_level": 3, "rekomendasi": True}
]

koleksi_menu.insert_many(data_menu)
print(f"Berhasil menginput data! Jumlah dokumen sekarang: {koleksi_menu.count_documents({})}")

Berhasil menginput data! Jumlah dokumen sekarang: 12


**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali.

In [6]:
# Query 1: Menu Kopi dengan tingkat_kepahitan > 3
print("--- 1. Menu Kopi dengan Tingkat Kepahitan > 3 ---")
query_1 = {"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}
for m in koleksi_menu.find(query_1):
    print(f"- {m['nama_menu']} (Kepahitan: {m['tingkat_kepahitan']})")

--- 1. Menu Kopi dengan Tingkat Kepahitan > 3 ---
- Espresso (Kepahitan: 5)
- Americano (Kepahitan: 4)
- Long Black (Kepahitan: 4)

--- 2. Menu dengan Harga antara 15.000 dan 30.000 ---


In [7]:
# Query 2: Harga >= 15000 dan <= 30000
print("\n--- 2. Menu dengan Harga antara 15.000 dan 30.000 ---")
query_2 = {"harga": {"$gte": 15000, "$lte": 30000}}
for m in koleksi_menu.find(query_2):
    print(f"- {m['nama_menu']} (Harga: Rp{m['harga']:,})")


--- 2. Menu dengan Harga antara 15.000 dan 30.000 ---
- Espresso (Harga: Rp18,000)
- Americano (Harga: Rp20,000)
- Kopi Susu Gula Aren (Harga: Rp22,000)
- Cafe Latte (Harga: Rp25,000)
- Cappuccino (Harga: Rp26,000)
- Long Black (Harga: Rp21,000)
- Mie Goreng Nusantara (Harga: Rp15,000)
- Nasi Goreng Gila (Harga: Rp28,000)
- Kentang Goreng Keju (Harga: Rp17,000)
- Roti Bakar Cokelat (Harga: Rp16,000)


In [8]:
# Query 3: Field pedas_level bernilai False pada pengecekan $existsprint("\n--- 3. Menu yang TIDAK memiliki field 'pedas_level' ---")
query_3 = {"pedas_level": {"$exists": False}}
for m in koleksi_menu.find(query_3):
    print(f"- {m['nama_menu']} (Kategori: {m['kategori']})")

- Espresso (Kategori: Kopi)
- Americano (Kategori: Kopi)
- Kopi Susu Gula Aren (Kategori: Kopi)
- Cafe Latte (Kategori: Kopi)
- Cappuccino (Kategori: Kopi)
- Long Black (Kategori: Kopi)


**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

In [10]:
# 1. Tambah field status: "Tersedia" ke semua dokumen
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})
print("Field 'status' berhasil ditambahkan ke semua menu.")

Field 'status' berhasil ditambahkan ke semua menu.


In [11]:
# 2. Ubah status menjadi "Habis" untuk 2 menu tertentu secara spesifik
koleksi_menu.update_one({"nama_menu": "Espresso"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Nasi Goreng Gila"}, {"$set": {"status": "Habis"}})
print("Status berhasil diubah menjadi 'Habis' untuk Espresso dan Nasi Goreng Gila.")

Status berhasil diubah menjadi 'Habis' untuk Espresso dan Nasi Goreng Gila.


In [12]:
# Verifikasi hasil update
print("\nVerifikasi status terbaru beberapa menu:")
for m in koleksi_menu.find({}, {"nama_menu": 1, "status": 1, "_id": 0}):
    print(f"- {m['nama_menu']}: {m['status']}")


Verifikasi status terbaru beberapa menu:
- Espresso: Habis
- Americano: Tersedia
- Kopi Susu Gula Aren: Tersedia
- Cafe Latte: Tersedia
- Cappuccino: Tersedia
- Long Black: Tersedia
- Mie Goreng Nusantara: Tersedia
- Nasi Goreng Gila: Habis
- Cireng Bumbu Rujak: Tersedia
- Kentang Goreng Keju: Tersedia
- Roti Bakar Cokelat: Tersedia
- Ayam Geprek Sambal Korek: Tersedia


**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.


In [17]:
from pyspark.sql.functions import avg
cursor_data = list(koleksi_menu.find({}))

df_pandas = pd.DataFrame(cursor_data)
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas = df_pandas.astype(object).where(pd.notnull(df_pandas), None)
df_spark_menu = spark.createDataFrame(df_pandas)


print("=== SKEMA DATA (printSchema) ===")
df_spark_menu.printSchema()

print("=== DATA MENU PADA SPARK DATAFRAME ===")
df_spark_menu.show()

print("=== RATA-RATA HARGA PER KATEGORI ===")
df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga")
).show()

=== SKEMA DATA (printSchema) ===
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: double (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)
 |-- rekomendasi: boolean (nullable = true)

=== DATA MENU PADA SPARK DATAFRAME ===
+--------------------+--------------------+-----+--------+-----------------+-----+--------+-----------+-----------+
|                 _id|           nama_menu|harga|kategori|tingkat_kepahitan| stok|  status|pedas_level|rekomendasi|
+--------------------+--------------------+-----+--------+-----------------+-----+--------+-----------+-----------+
|6ae33d0f54ecefca7...|            Espresso|18000|    Kopi|              5.0| 50.0|   Habis|       NULL|       NULL|
|6ae33d0f54ecefca7...|           Americano|20000|    Kopi|              4.0| 60.0|Tersedia|      

26/10/29 20:56:29 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

**E. Kesimpulan** *(bobot 20%)*

Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

Tantangan utama saat mengonversi data MongoDB semi-terstruktur menjadi Spark DataFrame yang bersifat kaku terletak pada ketidakseragaman baris data (flexible schema). Dokumen berkategori "Kopi" memiliki field tingkat_kepahitan namun absen dari field pedas_level, dan begitu pula sebaliknya pada kategori "Makanan". Ketika dikonversi langsung melalui Pandas ke Spark DataFrame, sistem database relasional/tabular dipaksa memetakan seluruh field yang ada ke dalam skema universal tunggal.
Dampaknya, baris yang tidak memiliki field tersebut secara otomatis diisi dengan nilai kosong (NaN di Pandas atau null di Spark DataFrame). Cara mengatasinya adalah dengan memastikan data dikonversi dengan tipe data homogen yang longgar (seperti string jika datanya kompleks) atau membiarkan Spark memetakan nilai tersebut sebagai tipe data angka yang dapat menampung nilai kosong (nullable), sehingga agregasi fungsi statistika seperti groupBy dan avg tetap dapat dieksekusi dengan aman tanpa menghasilkan galat struktural.

**EXPLORASI**

In [3]:
from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

spark = SparkSession.builder \
    .appName("EksplorasiBaru") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

cursor_data = list(koleksi_menu.find({}))
df_pandas = pd.DataFrame(cursor_data)
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas = df_pandas.astype(object).where(pd.notnull(df_pandas), None)
df_spark = spark.createDataFrame(df_pandas).fillna({"stok": 0})
df_hasil = df_spark.withColumn("total_nilai_aset", col("harga") * col("stok"))
df_hasil.select("nama_menu", "kategori", "harga", "stok", "total_nilai_aset").show()

+--------------------+--------+-----+-----+----------------+
|           nama_menu|kategori|harga| stok|total_nilai_aset|
+--------------------+--------+-----+-----+----------------+
|            Espresso|    Kopi|18000| 50.0|        900000.0|
|           Americano|    Kopi|20000| 60.0|       1200000.0|
| Kopi Susu Gula Aren|    Kopi|22000|100.0|       2200000.0|
|          Cafe Latte|    Kopi|25000| 40.0|       1000000.0|
|          Cappuccino|    Kopi|26000| 45.0|       1170000.0|
|          Long Black|    Kopi|21000| 30.0|        630000.0|
|Mie Goreng Nusantara| Makanan|15000|  0.0|             0.0|
|    Nasi Goreng Gila| Makanan|28000| 25.0|        700000.0|
|  Cireng Bumbu Rujak| Makanan|12000| 80.0|        960000.0|
| Kentang Goreng Keju| Makanan|17000| 55.0|        935000.0|
|  Roti Bakar Cokelat| Makanan|16000| 35.0|        560000.0|
|Ayam Geprek Samba...| Makanan|32000|  0.0|             0.0|
+--------------------+--------+-----+-----+----------------+



In [ ]:
spark.stop()
print("Spark Session sukses ditutup.")